# Tema: Spark SQL

## Objetivos
Practicar filtros, CTE, agregaciones y ventanas.

## Conceptos importantes para el examen
WHERE filtra filas; HAVING filtra grupos; IS NULL; UNION frente a UNION ALL.

**Dificultad:** Básico · **Tiempo estimado:** 50 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_03_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("overwrite").saveAsTable("employees")
display(employees.orderBy("employee_id"))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Filtro

In [ ]:
%sql
SELECT employee_id, name, salary FROM employees WHERE active ORDER BY salary DESC;

### 2. Agrupación

In [ ]:
%sql
SELECT department, COUNT(*) AS n, AVG(salary) AS average_salary FROM employees GROUP BY department HAVING COUNT(*) >= 5;

### 3. Ventana

In [ ]:
%sql
WITH r AS (SELECT *, ROW_NUMBER() OVER(PARTITION BY department ORDER BY salary DESC, employee_id) rn FROM employees)
SELECT name, department, salary FROM r WHERE rn = 1;

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Selecciona activos con salario entre 40.000 y 50.000 incluidos.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Clasifica a cada empleado como Data o Resto.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Obtén departamentos con salario medio mayor que 44.000.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Devuelve los dos mejor pagados de cada departamento, con desempate por ID.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Compara los recuentos de UNION y UNION ALL al combinar dos veces los ID 1–3.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** BETWEEN.

**Pista 2:** CASE WHEN.

**Pista 3:** HAVING.

**Pista 4:** CTE y ranking.

**Pista 5:** UNION elimina duplicados.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
%sql
SELECT * FROM employees WHERE active AND salary BETWEEN 40000 AND 50000;

### Solución 2

In [ ]:
%sql
SELECT employee_id, CASE WHEN department = 'Data' THEN 'Data' ELSE 'Resto' END AS area FROM employees;

### Solución 3

In [ ]:
%sql
SELECT department, AVG(salary) avg_salary FROM employees GROUP BY department HAVING AVG(salary) > 44000;

### Solución 4

In [ ]:
%sql
WITH r AS (SELECT *, ROW_NUMBER() OVER(PARTITION BY department ORDER BY salary DESC, employee_id) rn FROM employees)
SELECT employee_id, department, salary FROM r WHERE rn <= 2;

### Solución 5

In [ ]:
%sql
SELECT 'UNION' op, COUNT(*) n FROM (SELECT employee_id FROM employees WHERE employee_id <= 3 UNION SELECT employee_id FROM employees WHERE employee_id <= 3)
UNION ALL
SELECT 'UNION ALL', COUNT(*) FROM (SELECT employee_id FROM employees WHERE employee_id <= 3 UNION ALL SELECT employee_id FROM employees WHERE employee_id <= 3);

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Dónde filtras una suma agrupada?

A. ORDER BY

B. WHERE

C. HAVING

D. LIMIT

### Pregunta 2
¿Cómo detectas ausencia de salario?

A. salary = NULL

B. salary IS NULL

C. salary == 'NULL'

D. salary IN NULL

### Pregunta 3
¿Cómo conservas duplicados entre dos lotes?

A. UNION ALL

B. UNION

C. DISTINCT

D. INTERSECT

### Respuestas y explicación
**1. C** — HAVING evalúa grupos.

**2. B** — NULL requiere IS NULL.

**3. A** — Concatena sin deduplicar.

## PARTE 6 - RETO FINAL
Publica en Delta la masa salarial y el empleado mejor pagado por departamento entre los activos; usa CTE y ranking determinista.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
